In [ ]:
#| default_exp logger

# logger

> simple logger using idiomatic Solveit

In [ ]:
#| export
import json
from datetime import datetime
from html import unescape
from fastcore.all import patch
from dialoghelper.core import update_msg, read_msg, find_dname
from anyio.from_thread import BlockingPortalProvider

In [ ]:
import random
import IPython.display
from IPython.display import Markdown
import fastcore.all as FC
from fastcore.test import *
from anyio import sleep

In [ ]:
#| export
_provider = BlockingPortalProvider() # module level portal

In [ ]:
#| export
async def _update(msgid, output, dname): await update_msg(msgid, output=output, dname=dname)
async def _read(msgid, dname): return unescape((await read_msg(0, id=msgid, dname=dname)).output)
async def _id(): return (await read_msg(0)).id

In [ ]:
#| export
def _get_msg_id(): 
    with _provider as portal: return portal.call(_id)

In [ ]:
test_eq(_get_msg_id(), '_c0dac568')

In [ ]:
#| export
class Logger:
    "Timestamped logger that uses a cell's output as sink"
    def __init__(self, id:str='', dname:str='', clear:bool=True, sym:str='log', prepend:bool=False): 
        self.setup(id, dname, clear if not dname else False, sym, prepend)
        self.show()
    
    def setup(self, id:str='', dname:str='', clear:bool=False, sym:str='log', prepend:bool=False): 
        curr = find_dname()
        self.dname, self.msgid, self.prepend = dname or curr, id or _get_msg_id(), prepend
        self._xs = self.dname != curr
        with _provider as portal:
            if clear: self._s = ''; return portal.call(_update, self.msgid, '', self.dname)
            self._s = portal.call(_read, self.msgid, self.dname) if dname else getattr(self, '_s', '')
    
    def clear(self): 
        self._s = ''
        with _provider as portal: portal.call(_update, self.msgid, '', self.dname)
    
    @property
    def logs(self): return self._s.splitlines()
    def show(self): self.msgid = _get_msg_id(); print(self._s, end='')
    def __repr__(self): return self._s
    def __call__(self, *args, sep=' ', end='\n', file=None, flush=False): 
        dt = datetime.now()
        msg = sep.join(str(a) for a in args)
        s = f"[{dt:%H:%M:%S}.{dt.microsecond//1000:03d}] {msg}"
        self._s = (s + end + self._s) if self.prepend else (self._s + s + end)
        out = '[{"name": "stdout", "output_type": "stream", "text": %s}]' % json.dumps(self._s)
        with _provider as portal: portal.call(_update, self.msgid, out, self.dname)

In [ ]:
log = Logger()  # inject 'log'

[13:46:59.590] Some msg -> 819


In [ ]:
log('test')
log('test2')
log('test3')

In [ ]:
log(s := ';qwedcv fjkds')
test_is(s in log.logs[-1], True)
log.logs

['[13:46:43.427] test',
 '[13:46:43.471] test2',
 '[13:46:43.504] test3',
 '[13:46:47.365] ;qwedcv fjkds']

In [ ]:
print(log, end='')  # __str__

[13:46:43.427] test
[13:46:43.471] test2
[13:46:43.504] test3
[13:46:47.365] ;qwedcv fjkds


In [ ]:
print(log._s, end='')

[13:46:43.427] test
[13:46:43.471] test2
[13:46:43.504] test3
[13:46:47.365] ;qwedcv fjkds


In [ ]:
log # __repr__

[13:46:43.427] test
[13:46:43.471] test2
[13:46:43.504] test3
[13:46:47.365] ;qwedcv fjkds

In [ ]:
log.clear()
test_eq(log.logs, [])

In [ ]:
log(f"Some msg -> {random.randint(0, 1000)}")
log.logs

['[13:46:59.590] Some msg -> 819']

In [ ]:
log.show()

In [ ]:
#| export
@patch
def show(self:Logger, clear:bool=False):
    "Display log in current cell, optionally clearing first"
    if self._xs: 
        with _provider as portal: self._s = portal.call(_read, self.msgid, self.dname)
    else:
        if self.msgid != _get_msg_id():
            oldid = self.msgid
            self.setup()
            with _provider as portal: portal.call(_update, oldid, '', self.dname)
        if clear: self.clear()
    print(self._s, end='')

In [ ]:
log.show()

In [ ]:
log.show()

In [ ]:
for _ in "En algún lugar de la Mancha".split(): log(_)

In [ ]:
# across target
log.show()

[13:46:59.590] Some msg -> 819
[13:47:05.635] En
[13:47:05.668] algún
[13:47:05.700] lugar
[13:47:05.733] de
[13:47:05.766] la
[13:47:05.803] Mancha


Look at `show` method. What does it do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `show` method does three things:

1. **Checks if it's being called from a different cell than where the logger was last set up.** If so, it moves the logger to the new cell (updates `self.msgid` to the current cell) and clears the output from the old cell.

2. **Optionally clears the log** if `clear=True` is passed.

3. **Prints the current log content** (`self._s`) to the current cell's output.

This lets you call `log.show()` from any cell to display the accumulated logs there, and the logger will automatically "move" its display to that new cell while cleaning up the previous one.

Look at the `Logger` class and all the examples. What does it do?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `Logger` class creates a simple logging system that displays timestamped messages in a cell's output area. Here's what it does:

**Core functionality:**
- Logs messages with timestamps (down to milliseconds)
- Stores messages in reverse chronological order (newest first)
- Updates a specific message cell's output in real-time as you add log entries

**Key features:**
- `log('message')` adds a timestamped entry
- `log.clear()` wipes all entries
- `log.show()` displays the log (and handles moving between cells)
- Messages are kept in `log.logs` as a list

**The clever bit:** It uses `update_msg()` to write directly to a cell's output, so you can log from anywhere in your notebook and see all the messages accumulate in one place. Notice how in the examples, multiple `log()` calls build up a running list with timestamps.

What aspect would you like to explore more? The timestamp formatting, how it tracks which cell to update, or something else?

# Next

In [ ]:
print??

Logger is not designed to be a full replacement of `print`. For instance, currently doesn't honor `sep`, `end` or any other argument. Think about we should do to get parity with `print` starting with `end` > `sep` > `file` > `flush`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The current `__call__` already has the right signature but ignores those args. Here's how I'd think about each:

- **`sep`** — easiest, just pass it to the `str.join()` call that builds `msg`. Currently hardcoded to `' '`.

- **`end`** — trickier. Right now a `\n` is always inserted between entries to build the reverse-ordered stack. Honoring `end=''` (the common use case, e.g. progress updates) would mean the current entry *merges* with the next one rather than being on its own line. We'd need to decide: does `end` affect how entries are *separated in the log*, or just the final `print` output? Probably the latter is safer — i.e. `end` controls the trailing character appended to `msg` before it gets prepended to `self._s`.

- **`file`** — if a `file` is provided, we bypass the cell-output mechanism entirely and just write to that stream. A simple early-return branch would handle it.

- **`flush`** — essentially a no-op for us since we push to the cell on every call anyway, but we could call `file.flush()` when `file` is provided.

The most interesting design question is `end`: do you want `end=''` to suppress the newline *between log entries*, or just affect what gets appended to the message string itself?

It's more than that. What about ansi codes (including control). For instance, besides colors, `rich` makes good use of them to simulate addresing lines. `end` is only a poor-man ansi code that only affects the current line. I think is safe to assume what `print` docs say: "string appended after the last value, default a newline". But the real conumdrum is how to handle ansi codes.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a really important point. ANSI codes fall into two broad categories relevant here:

**Style codes** (colors, bold, etc.) — these are relatively safe. They don't change the logical structure of the text, just its appearance. The cell output renders them fine (Jupyter/solveit supports ANSI in stream outputs).

**Control codes** (cursor movement, line clearing, `\r`, etc.) — these are the real problem. Tools like `rich` use them to simulate in-place updates (e.g. progress bars overwriting a line with `\r` or `\033[1A` to move up). Our logger fundamentally can't honor these because:

1. We're storing `self._s` as a plain string and rebuilding the whole output on every call — there's no concept of a "cursor position" in our model.
2. The cell output isn't a real terminal; it's rendered HTML. Control codes that work in a TTY may be ignored or mangled.
3. Our reverse-chronological prepending model is at odds with in-place update patterns entirely.

**The conundrum then:** do we want to support `rich`-style usage at all, or explicitly document it as out of scope? One option is to detect/strip control codes (e.g. via a regex on `\033[...`) before storing, so at least style codes survive but control codes don't corrupt `self._s`. Another is to simply pass through to a real stream when `file` is set, leaving that problem to the caller.

What's your intuition — strip and warn, or just document the limitation?

WRT to prepending modes, make it an option. Ansi style codes are transparent, we just add a new property `clean_logs` that returns a stripped version of `_s`. Control codes, I don't know. On the one hand, it's cool, but on the other, it seems complex with Solveit's current capabilities.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good plan. Three concrete changes then:

1. **`prepend` param** on `__init__`/`setup` — when `False`, new entries are appended instead of prepended to `self._s`.
2. **`clean_logs` property** — strips ANSI SGR codes (style only, e.g. `\033[...m`) via regex, returns a list like `logs`.
3. **Control codes** — leave for now, document as unsupported.

Let's start with `prepend`, patching `__init__` and `__call__`:

```python
@patch
def __init__(self:Logger, id:str='', dname:str='', clear:bool=True, sym:str='log', prepend:bool=True):
    self.prepend = prepend
    self.setup(id, dname, clear if not dname else False, sym)
    self.show()
```

```python
@patch
def __call__(self:Logger, *args, sep=' ', end='\n', file=None, flush=False):
    "Log a timestamped message"
    dt = datetime.now()
    msg = sep.join(str(a) for a in args) + (end if end != '\n' else '')
    s = f"[{dt:%H:%M:%S}.{dt.microsecond//1000:03d}] {msg}"
    self._s = (s + (f"\n{self._s}" if self._s else '')) if self.prepend else (f"{self._s}\n{s}" if self._s else s)
    out = '[{"name": "stdout", "output_type": "stream", "text": %s}]' % json.dumps(self._s)
    with _provider as portal: portal.call(_update, self.msgid, out, self.dname)
```

Then the `clean_logs` property:

```python
_ansi_re = re.compile(r'\033\[[0-9;]*m')

@patch
@property
def clean_logs(self:Logger): return [_ansi_re.sub('', l) for l in self.logs]
```

# export -

In [ ]:
from dutil.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from dutil.core import dlg_export
# dlg_export()